# MCP (Model Context Protocol) for Computational Toxicology
### Building AI-Powered Toxicology Assistants — A Complete Step-by-Step Tutorial

**Author:** Himanshu Goel | [hgoelgithub.github.io](https://hgoelgithub.github.io)

---

## What is MCP and why does it transform toxicology workflows?

**MCP (Model Context Protocol)** is an open standard (Anthropic, 2024) that
connects AI models like Claude to external tools, databases, and scientific
software through a single, standardised interface.

```
Before MCP                        After MCP
──────────────────────────────    ──────────────────────────────
Scientist writes Python code      Scientist asks Claude:
to call PubChem API               'Screen these 50 compounds
Scientist writes more code         for ICH M7 genotoxicity'
to run RDKit descriptors
Scientist writes more code        Claude calls your MCP tools,
to apply ICH M7 alerts            synthesises results, flags
Scientist writes report           concerns, generates report
    → Hours of work               → Minutes, zero extra code
```

## MCP in computational toxicology — the key insight

Your existing toxicology functions (RDKit, ADMET models, structural alert
screeners, IVIVE calculators) become **callable tools** for any AI agent.
MCP is the bridge:

```
Claude (AI agent)
       │
       │  MCP protocol (JSON-RPC over STDIO or HTTP)
       │
┌──────┴──────────────────────────────────────────────┐
│           Your MCP Toxicology Server                 │
│                                                      │
│  compute_admet()    screen_ich_m7()   predict_herg() │
│  compute_ivive()    assess_dili()     query_pubchem() │
│  generate_report()  run_vina()        search_chembl() │
└──────────────────────────────────────────────────────┘
```

## Tutorial sections

| # | Section | What you build |
|---|---------|----------------|
| 1 | Setup and concepts | pip install, core MCP vocabulary |
| 2 | First MCP server | Hello-world toxicology server in 30 lines |
| 3 | RDKit as MCP tools | SMILES → ADMET, alerts, similarity |
| 4 | Databases as MCP resources | Toxicology DB with URI addressing |
| 5 | ICH M7 two-method assessment | Structural alerts + QSAR via MCP |
| 6 | IVIVE and ADMET pipeline | EPA HTTK dose estimation |
| 7 | Complete ADMET server | 8 endpoints, one server file |
| 8 | Connecting Claude Desktop | Config file + live demo |
| 9 | Anthropic API tool-use pattern | Automated screening pipeline |
| 10 | Production patterns & cheatsheet | Error handling, best practices |

---
## Section 1 — Setup and Core Concepts

### pip-only installation

```bash
python -m venv ~/envs/mcp_tox
source ~/envs/mcp_tox/bin/activate
pip install --upgrade pip

# MCP SDK
pip install mcp

# Toxicology / cheminformatics
pip install rdkit requests pandas numpy matplotlib

# Optional: HTTP transport
pip install fastapi uvicorn httpx

# VS Code kernel
pip install ipykernel
python -m ipykernel install --user --name=mcp_tox --display-name='Python (mcp_tox)'
```

### The four MCP primitives

| Primitive | What it is | Toxicology example |
|-----------|-----------|-------------------|
| **Tool** | Function the AI calls (active) | `compute_admet(smiles)` |
| **Resource** | Data the AI reads (passive) | `tox://compound/aspirin` |
| **Prompt** | Template the AI uses | `iata_assessment_prompt` |
| **Sampling** | AI asks server to generate text | Summarise a long SDS sheet |

### Two transport modes

```
STDIO (default — local tools)
  Claude ──stdin──▶ your_server.py ──stdout──▶ Claude
  • Subprocess launched by Claude Desktop
  • Perfect for local RDKit, file system access

HTTP / SSE (network — team servers)
  Claude ──HTTP──▶ http://localhost:8000/sse ──SSE──▶ Claude
  • Any client on the network can connect
  • Use for shared ADMET model servers, databases
```

### MCP message format (JSON-RPC 2.0)

```json
// Claude sends:
{ "jsonrpc": "2.0", "id": 1,
  "method": "tools/call",
  "params": { "name": "compute_admet",
               "arguments": { "smiles": "c1ccccc1" } } }

// Your server responds:
{ "jsonrpc": "2.0", "id": 1,
  "result": { "content": [
    { "type": "text",
      "text": "{\"MW\": 78.11, \"LogP\": 1.69, \"QED\": 0.43}" }
  ] } }
```

In [ ]:
# ── Section 1: Check environment and imports ─────────────────────────────────
import os, subprocess, sys, warnings
warnings.filterwarnings('ignore')

def check(name, import_as=None):
    try:
        m = __import__(import_as or name)
        v = getattr(m, '__version__', 'installed')
        return f'OK  ({v})'
    except ImportError:
        return 'MISSING'

print('Dependency check:')
for pkg, imp in [('mcp','mcp'), ('rdkit','rdkit'), ('requests','requests'),
                  ('fastapi','fastapi'), ('pandas','pandas'), ('numpy','numpy')]:
    print(f'  {pkg:12s}: {check(pkg, imp)}')

# Show MCP package structure if available
try:
    import mcp
    print(f'\nMCP package version: {mcp.__version__}')
    print('Key classes available:')
    print('  mcp.server.Server           -- creates an MCP server')
    print('  mcp.server.stdio            -- STDIO transport')
    print('  mcp.types.Tool              -- declare a tool')
    print('  mcp.types.TextContent       -- tool response')
    print('  mcp.types.Resource          -- declare a resource')
except ImportError:
    print('\nInstall MCP: pip install mcp')

print()
# Create working directory
os.makedirs('mcp_servers', exist_ok=True)
print('Working directory: mcp_servers/')

---
## Section 2 — Your First Toxicology MCP Server (30 lines)

The minimal MCP server pattern has four parts:

```
1.  server = Server('my-server')          create the server
2.  @server.list_tools()                  declare what tools exist
3.  @server.call_tool()                   handle when a tool is called
4.  stdio_server() + server.run()         start the server loop
```

That is the entire pattern. Everything else is just your science functions.

**Step by step:**
1. Save the file to `~/airflow/dags/` — wait, wrong tutorial 😄
2. Save the server file anywhere on disk
3. Register it in `claude_desktop_config.json`
4. Claude Desktop picks it up automatically on restart

In [ ]:
# ── Write the first server to disk ──────────────────────────────────────────
import os

server_code = '''#!/usr/bin/env python3
# mcp_servers/hello_tox.py -- minimal toxicology MCP server
# Run directly: python mcp_servers/hello_tox.py
# Register in Claude Desktop: see Section 8

from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp import types
import asyncio

# ── Step 1: Create server ────────────────────────────────────────────────
server = Server('hello-toxicology')

# ── Step 2: Declare tools ────────────────────────────────────────────────
@server.list_tools()
async def list_tools():
    return [
        types.Tool(
            name='check_lipinski',
            description=('Check if a molecule passes Lipinski Rule of Five '
                         'for oral bioavailability. Input: SMILES string.'),
            inputSchema={
                'type': 'object',
                'properties': {
                    'smiles': {'type': 'string', 'description': 'SMILES string'}
                },
                'required': ['smiles']
            }
        ),
    ]

# ── Step 3: Handle tool calls ────────────────────────────────────────────
@server.call_tool()
async def call_tool(name: str, arguments: dict):
    if name == 'check_lipinski':
        result = lipinski_check(arguments['smiles'])
        import json
        return [types.TextContent(type='text', text=json.dumps(result, indent=2))]
    raise ValueError(f'Unknown tool: {name}')

# ── Science function (completely separate from MCP plumbing) ─────────────
def lipinski_check(smiles: str) -> dict:
    try:
        from rdkit import Chem
        from rdkit.Chem import Descriptors, rdMolDescriptors
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return {'error': f'Invalid SMILES: {smiles}'}
        mw  = Descriptors.MolWt(mol)
        lp  = Descriptors.MolLogP(mol)
        hbd = rdMolDescriptors.CalcNumHBD(mol)
        hba = rdMolDescriptors.CalcNumHBA(mol)
        violations = sum([mw > 500, lp > 5, hbd > 5, hba > 10])
        return {
            'smiles':     smiles,
            'MW':         round(mw, 2),
            'LogP':       round(lp, 3),
            'HBD':        hbd,
            'HBA':        hba,
            'violations': violations,
            'passes_Ro5': violations <= 1,
            'verdict':    'PASS' if violations <= 1 else 'FAIL',
        }
    except Exception as e:
        return {'error': str(e)}

# ── Step 4: Run server ───────────────────────────────────────────────────
async def main():
    async with stdio_server() as (r, w):
        await server.run(r, w, server.create_initialization_options())

if __name__ == '__main__':
    asyncio.run(main())
'''

with open('mcp_servers/hello_tox.py', 'w') as f:
    f.write(server_code)
print('Saved: mcp_servers/hello_tox.py')
print()
print('Key points:')
notes = [
    'Server() -- creates a named MCP server',
    '@server.list_tools() -- tells clients what tools exist',
    '@server.call_tool() -- runs when Claude calls a tool',
    'Science function is just a normal Python function',
    'stdio_server() -- STDIO transport (simplest for local use)',
    'asyncio.run(main()) -- starts the blocking server loop',
]
for n in notes:
    print(f'  {n}')

print()
print('Test the science function directly:')

# Test without MCP infrastructure
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, rdMolDescriptors

    def lipinski_check_demo(smiles):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return {'error': 'Invalid SMILES'}
        mw  = Descriptors.MolWt(mol)
        lp  = Descriptors.MolLogP(mol)
        hbd = rdMolDescriptors.CalcNumHBD(mol)
        hba = rdMolDescriptors.CalcNumHBA(mol)
        v   = sum([mw>500, lp>5, hbd>5, hba>10])
        return {'MW': round(mw,2), 'LogP': round(lp,3), 'HBD': hbd,
                'HBA': hba, 'violations': v, 'passes_Ro5': v<=1}

    for name, smi in [('Aspirin','CC(=O)Oc1ccccc1C(=O)O'),
                       ('Paclitaxel','O=C(OC1C(OC(=O)c2ccccc2)C2(O)CC(OC(=O)C(O)c3ccccc3)CC(C)(C3CC(OC(=O)c4ccccc4)C(=O)O3)C2C1C)C')]:
        r = lipinski_check_demo(smi)
        print(f'  {name}: Ro5={r["passes_Ro5"]}  MW={r["MW"]}  violations={r["violations"]}')
except ImportError:
    print('  (RDKit not installed -- pip install rdkit)')

---
## Section 3 — RDKit Cheminformatics as MCP Tools

This section builds a server exposing four production-quality RDKit tools.
Each follows the same pattern: **one Python function = one MCP tool**.

| Tool | Science | Regulatory |
|------|---------|------------|
| `compute_admet` | MW, LogP, TPSA, HBD, HBA, Fsp3, QED, Ro5 | Oral BA screening |
| `screen_structural_alerts` | 8 ICH M7 SMARTS patterns | ICH M7(R2) Method 1 |
| `screen_pains` | RDKit FilterCatalog | Pan-assay interference |
| `compute_tanimoto` | ECFP4 Tanimoto | Read-across, analogue search |

### Why good tool descriptions matter

Claude decides **which tool to call** based solely on the `description` field.
A vague description (`'Does stuff with molecules'`) will never be called.
A precise description tells Claude exactly what the tool does and when to use it.

In [ ]:
# ── Define the science functions (pure Python, no MCP dependency) ────────────
# These can be tested directly in the notebook before wrapping as MCP tools.

import warnings; warnings.filterwarnings('ignore')
import json

try:
    from rdkit import Chem
    from rdkit.Chem import (Descriptors, rdMolDescriptors, AllChem,
                              QED, DataStructs)
    from rdkit.Chem.FilterCatalog import FilterCatalogParams, FilterCatalog
    RDKIT_OK = True
except ImportError:
    RDKIT_OK = False

# ── Function 1: ADMET properties ─────────────────────────────────────────────
def compute_admet(smiles: str) -> dict:
    mol = Chem.MolFromSmiles(smiles) if RDKIT_OK else None
    if mol is None: return {'error': f'Invalid SMILES: {smiles}'}
    mw   = Descriptors.MolWt(mol)
    lp   = Descriptors.MolLogP(mol)
    tpsa = Descriptors.TPSA(mol)
    hbd  = rdMolDescriptors.CalcNumHBD(mol)
    hba  = rdMolDescriptors.CalcNumHBA(mol)
    rb   = rdMolDescriptors.CalcNumRotatableBonds(mol)
    fsp3 = rdMolDescriptors.CalcFractionCSP3(mol)
    qed  = QED.qed(mol)
    viol = sum([mw > 500, lp > 5, hbd > 5, hba > 10])
    return {
        'MW': round(mw, 2), 'LogP': round(lp, 3), 'TPSA': round(tpsa, 1),
        'HBD': hbd, 'HBA': hba, 'RotBonds': rb,
        'Fsp3': round(fsp3, 3), 'QED': round(qed, 3),
        'Ro5_violations': viol, 'oral_BA': 'likely' if viol <= 1 else 'poor',
        'BBB_estimate': 'CNS-penetrant' if tpsa < 90 and 1 < lp < 3 and mw < 400
                        else 'non-CNS',
    }

# ── Function 2: ICH M7 structural alerts ─────────────────────────────────────
ICH_M7_ALERTS = {
    'Nitrosamine':      '[N;!$(N=O)]-N=O',
    'Aromatic_nitro':   'c[N+](=O)[O-]',
    'Aliphatic_nitro':  'C[N+](=O)[O-]',
    'Aromatic_amine':   '[NH2]c',
    'Michael_acceptor': '[$(C=CC=O),$(C=CS)]',
    'Epoxide':          '[C;R0]1OC1',
    'Hydrazine':        '[NH2]N',
    'Diazonium':        '[#6][N+]#N',
}

def screen_structural_alerts(smiles: str) -> dict:
    mol = Chem.MolFromSmiles(smiles) if RDKIT_OK else None
    if mol is None: return {'error': f'Invalid SMILES: {smiles}'}
    hits = []
    for name, smarts in ICH_M7_ALERTS.items():
        patt = Chem.MolFromSmarts(smarts)
        if patt and mol.HasSubstructMatch(patt):
            hits.append(name)
    class1 = [h for h in hits if 'Nitrosamine' in h]
    return {
        'alerts_found': hits,
        'n_alerts':     len(hits),
        'ich_m7_class': ('Class 1 (known human mutagen)' if class1
                         else 'Class 2 (alert present)' if hits
                         else 'Class 5 (no alert)'),
        'genotox_concern': len(hits) > 0,
        'recommendation': ('Do not progress' if class1
                           else 'In vitro genotox testing required' if hits
                           else 'No further genotox testing needed'),
    }

# ── Function 3: PAINS screen ─────────────────────────────────────────────────
def screen_pains(smiles: str) -> dict:
    mol = Chem.MolFromSmiles(smiles) if RDKIT_OK else None
    if mol is None: return {'error': f'Invalid SMILES: {smiles}'}
    params = FilterCatalogParams()
    params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
    catalog = FilterCatalog(params)
    entry   = catalog.GetFirstMatch(mol)
    if entry:
        return {'pains_alert': True,  'pattern': entry.GetDescription(),
                'recommendation': 'Flag as likely assay interference'}
    return {'pains_alert': False, 'status': 'Clean — no PAINS detected'}

# ── Function 4: Tanimoto similarity ──────────────────────────────────────────
def compute_tanimoto(smiles1: str, smiles2: str) -> dict:
    m1 = Chem.MolFromSmiles(smiles1) if RDKIT_OK else None
    m2 = Chem.MolFromSmiles(smiles2) if RDKIT_OK else None
    if not m1 or not m2: return {'error': 'One or both SMILES invalid'}
    fp1 = AllChem.GetMorganFingerprintAsBitVect(m1, 2, 2048)
    fp2 = AllChem.GetMorganFingerprintAsBitVect(m2, 2, 2048)
    tc  = DataStructs.TanimotoSimilarity(fp1, fp2)
    return {
        'tanimoto': round(tc, 4),
        'interpretation': ('Identical' if tc > 0.99 else
                           'Very similar' if tc > 0.85 else
                           'Similar'     if tc > 0.60 else
                           'Related'     if tc > 0.40 else
                           'Dissimilar'),
        'read_across_candidate': tc >= 0.40,
    }

# ── Test all functions before wrapping as MCP tools ──────────────────────────
if RDKIT_OK:
    aspirin = 'CC(=O)Oc1ccccc1C(=O)O'
    ndma    = 'CN(C)N=O'
    print('compute_admet (aspirin):')
    print(json.dumps(compute_admet(aspirin), indent=2))
    print()
    print('screen_structural_alerts (NDMA):')
    print(json.dumps(screen_structural_alerts(ndma), indent=2))
    print()
    print('screen_pains (aspirin):')
    print(json.dumps(screen_pains(aspirin), indent=2))
    print()
    print('compute_tanimoto (aspirin vs ibuprofen):')
    print(json.dumps(compute_tanimoto(aspirin, 'CC(C)Cc1ccc(cc1)C(C)C(=O)O'), indent=2))
else:
    print('RDKit not available — pip install rdkit')

In [ ]:
# ── Write the complete RDKit MCP server ──────────────────────────────────────

server_src = '''#!/usr/bin/env python3
# mcp_servers/rdkit_tox_server.py
# Run: python mcp_servers/rdkit_tox_server.py

from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp import types
import asyncio, json
from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, QED
from rdkit.Chem.FilterCatalog import FilterCatalogParams, FilterCatalog

server = Server('rdkit-toxicology')

ALERTS = {
    'Nitrosamine':    '[N;!$(N=O)]-N=O',
    'Aromatic_nitro': 'c[N+](=O)[O-]',
    'Aromatic_amine': '[NH2]c',
    'Michael_acc':    '[$(C=CC=O),$(C=CS)]',
    'Epoxide':        '[C;R0]1OC1',
}

@server.list_tools()
async def list_tools():
    return [
        types.Tool(
            name='compute_admet',
            description=('Compute ADMET physicochemical properties from SMILES: '
                         'MW, LogP, TPSA, HBD, HBA, Fsp3, QED, Ro5 violations, '
                         'oral bioavailability estimate, BBB penetration estimate. '
                         'Replaces rat oral bioavailability study.'),
            inputSchema={'type':'object','properties':{'smiles':{'type':'string'}},
                         'required':['smiles']}
        ),
        types.Tool(
            name='screen_alerts',
            description=('Screen molecule for ICH M7(R2) structural alerts '
                         '(genotoxic impurity assessment). Returns ICH class '
                         '1-5 and recommendation. Implements Method 1 of '
                         'two-method framework.'),
            inputSchema={'type':'object','properties':{'smiles':{'type':'string'}},
                         'required':['smiles']}
        ),
        types.Tool(
            name='screen_pains',
            description='Screen for PAINS (pan-assay interference compounds). Flags likely false positives in HTS assays.',
            inputSchema={'type':'object','properties':{'smiles':{'type':'string'}},'required':['smiles']}
        ),
        types.Tool(
            name='tanimoto',
            description='Compute Tanimoto similarity (ECFP4) between two molecules for read-across assessment.',
            inputSchema={'type':'object','properties':{'smiles1':{'type':'string'},'smiles2':{'type':'string'}},'required':['smiles1','smiles2']}
        ),
    ]

@server.call_tool()
async def call_tool(name: str, arguments: dict):
    try:
        if name == 'compute_admet':
            r = _admet(arguments['smiles'])
        elif name == 'screen_alerts':
            r = _alerts(arguments['smiles'])
        elif name == 'screen_pains':
            r = _pains(arguments['smiles'])
        elif name == 'tanimoto':
            r = _tanimoto(arguments['smiles1'], arguments['smiles2'])
        else:
            r = {'error': f'Unknown tool: {name}'}
        return [types.TextContent(type='text', text=json.dumps(r, indent=2))]
    except Exception as e:
        return [types.TextContent(type='text', text=json.dumps({'error': str(e)}))]

def _admet(smi):
    mol = Chem.MolFromSmiles(smi)
    if not mol: return {'error': f'Invalid: {smi}'}
    mw=Descriptors.MolWt(mol); lp=Descriptors.MolLogP(mol)
    tpsa=Descriptors.TPSA(mol); hbd=rdMolDescriptors.CalcNumHBD(mol)
    hba=rdMolDescriptors.CalcNumHBA(mol); qed=round(QED.qed(mol),3)
    v=sum([mw>500,lp>5,hbd>5,hba>10])
    return {'MW':round(mw,2),'LogP':round(lp,3),'TPSA':round(tpsa,1),
            'HBD':hbd,'HBA':hba,'QED':qed,'Ro5_violations':v,
            'oral_BA':'likely' if v<=1 else 'poor'}

def _alerts(smi):
    mol=Chem.MolFromSmiles(smi)
    if not mol: return {'error':f'Invalid: {smi}'}
    hits=[k for k,v in ALERTS.items() if Chem.MolFromSmarts(v) and mol.HasSubstructMatch(Chem.MolFromSmarts(v))]
    c1=[h for h in hits if 'Nitrosamine' in h]
    return {'alerts':hits,'n_alerts':len(hits),
            'ich_m7_class':'Class 1' if c1 else 'Class 2' if hits else 'Class 5',
            'genotox_concern':len(hits)>0}

def _pains(smi):
    mol=Chem.MolFromSmiles(smi)
    if not mol: return {'error':f'Invalid: {smi}'}
    p=FilterCatalogParams(); p.AddCatalog(p.FilterCatalogs.PAINS)
    e=FilterCatalog(p).GetFirstMatch(mol)
    return {'pains':True,'pattern':e.GetDescription()} if e else {'pains':False}

def _tanimoto(s1,s2):
    m1,m2=Chem.MolFromSmiles(s1),Chem.MolFromSmiles(s2)
    if not m1 or not m2: return {'error':'Invalid SMILES'}
    fp1=AllChem.GetMorganFingerprintAsBitVect(m1,2,2048)
    fp2=AllChem.GetMorganFingerprintAsBitVect(m2,2,2048)
    return {'tanimoto':round(DataStructs.TanimotoSimilarity(fp1,fp2),4)}

async def main():
    async with stdio_server() as (r,w):
        await server.run(r,w,server.create_initialization_options())

if __name__=='__main__':
    asyncio.run(main())
'''

with open('mcp_servers/rdkit_tox_server.py', 'w') as f:
    f.write(server_src)
print('Saved: mcp_servers/rdkit_tox_server.py')
print('Tools: compute_admet | screen_alerts | screen_pains | tanimoto')

---
## Section 4 — Toxicology Databases as MCP Resources

**MCP Resources** are read-only data the AI can access — files, databases,
API responses. They use **URI addressing**, like web URLs:

```
tox://compound/aspirin           Full toxicology profile
tox://compound/cid:2244          By PubChem CID
tox://database/summary           List all compounds
tox://assay/ames/aspirin         Ames result for aspirin
```

### Resources vs Tools

```
Tools     → AI calls them    → return computed results  (active)
Resources → AI reads them    → return stored data       (passive)
```

Resources are ideal for:
- Reference toxicology databases (DILIrank, ChemIDplus)
- Regulatory guideline text (ICH M7, OECD TG)
- Pre-computed ADMET tables
- Literature reference sets

In [ ]:
# ── Build a reference toxicology database ────────────────────────────────────
# 8 well-characterised compounds with experimental values

TOX_DB = {
    'aspirin': {
        'name':'Aspirin', 'cas':'50-78-2', 'smiles':'CC(=O)Oc1ccccc1C(=O)O',
        'mw':180.16, 'logp':1.19, 'ames':'Negative',
        'dili_rank':'Less-concern', 'herg_risk':'Low',
        'ld50_rat_mg_kg':200, 'ghs_class':'Category 4',
        'class':'NSAID analgesic',
    },
    'diclofenac': {
        'name':'Diclofenac', 'cas':'15307-86-5', 'smiles':'O=C(O)Cc1ccccc1Nc1c(Cl)cccc1Cl',
        'mw':296.15, 'logp':3.96, 'ames':'Negative',
        'dili_rank':'Most-concern', 'herg_risk':'Medium',
        'ld50_rat_mg_kg':150, 'ghs_class':'Category 4',
        'class':'NSAID analgesic',
    },
    'caffeine': {
        'name':'Caffeine', 'cas':'58-08-2', 'smiles':'Cn1cnc2c1c(=O)n(C)c(=O)n2C',
        'mw':194.19, 'logp':-0.07, 'ames':'Negative',
        'dili_rank':'No-concern', 'herg_risk':'Low',
        'ld50_rat_mg_kg':367, 'ghs_class':'Category 5',
        'class':'Stimulant / Xanthine',
    },
    'cisapride': {
        'name':'Cisapride', 'cas':'81098-60-4', 'smiles':'COCCNC(=O)c1cc(Cl)c(N)cc1OC1CCNCC1',
        'mw':465.95, 'logp':2.88, 'ames':'Negative',
        'dili_rank':'Most-concern', 'herg_risk':'HIGH (withdrawn)',
        'ld50_rat_mg_kg':200, 'ghs_class':'Category 4',
        'class':'Prokinetic (withdrawn — cardiac risk)',
    },
    'ndma': {
        'name':'NDMA (N-Nitrosodimethylamine)', 'cas':'62-75-9', 'smiles':'CN(C)N=O',
        'mw':74.08, 'logp':-0.57, 'ames':'Positive',
        'dili_rank':'N/A', 'herg_risk':'Low',
        'ld50_rat_mg_kg':27, 'ghs_class':'Category 2',
        'iarc':'Group 2A (probable human carcinogen)',
        'ich_m7_class':'Class 1 (known human mutagen)',
        'acceptable_intake_ng_day': 0.096,
        'class':'Nitrosamine contaminant',
    },
    'metformin': {
        'name':'Metformin', 'cas':'657-24-9', 'smiles':'CN(C)C(=N)NC(=N)N',
        'mw':129.16, 'logp':-1.43, 'ames':'Negative',
        'dili_rank':'No-concern', 'herg_risk':'Low',
        'ld50_rat_mg_kg':1000, 'ghs_class':'Category 5',
        'class':'Antidiabetic (biguanide)',
    },
    'atorvastatin': {
        'name':'Atorvastatin', 'cas':'134523-00-5',
        'smiles':'CC(C)c1c(C(=O)Nc2ccccc2F)c(-c2ccccc2)n(CC[C@@H](O)C[C@@H](O)CC(=O)O)c1-c1ccc(F)cc1',
        'mw':558.64, 'logp':4.5, 'ames':'Negative',
        'dili_rank':'Less-concern', 'herg_risk':'Medium',
        'ld50_rat_mg_kg':34, 'ghs_class':'Category 4',
        'class':'Statin (HMG-CoA reductase inhibitor)',
    },
    'tamoxifen': {
        'name':'Tamoxifen', 'cas':'10540-29-1',
        'smiles':'CC/C(=C(\\c1ccccc1)/c1ccc(OCCN(C)C)cc1)c1ccccc1',
        'mw':371.51, 'logp':6.5, 'ames':'Positive',
        'dili_rank':'Most-concern', 'herg_risk':'HIGH',
        'ld50_rat_mg_kg':338, 'ghs_class':'Category 4',
        'class':'SERM / anticancer',
    },
}

print(f'Database: {len(TOX_DB)} reference compounds')
print()
for k, v in TOX_DB.items():
    print(f'  {k:15s}: DILI={v["dili_rank"]:12s}  '
          f'Ames={v["ames"]:8s}  hERG={v["herg_risk"]}')

In [ ]:
# ── MCP server with Resources + search Tools ─────────────────────────────────

resource_server = '''#!/usr/bin/env python3
# mcp_servers/tox_database_server.py
# Exposes toxicology database as MCP resources + search tools

from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp import types
import asyncio, json

server = Server('tox-database')

# Inline database (production: load from SQLite or PostgreSQL)
DB = {
    'aspirin':   {'name':'Aspirin','smiles':'CC(=O)Oc1ccccc1C(=O)O','ames':'Negative','dili':'Less-concern'},
    'ndma':      {'name':'NDMA','smiles':'CN(C)N=O','ames':'Positive','dili':'N/A','ich_m7':'Class 1'},
    'caffeine':  {'name':'Caffeine','smiles':'Cn1cnc2c1c(=O)n(C)c(=O)n2C','ames':'Negative','dili':'No-concern'},
    'cisapride': {'name':'Cisapride','smiles':'COCCNC(=O)c1cc(Cl)c(N)cc1OC1CCNCC1','ames':'Negative','dili':'Most-concern'},
}

# ── Resources: read-only data at URI addresses ───────────────────────────
@server.list_resources()
async def list_resources():
    resources = [
        types.Resource(uri='tox://database/summary',
                       name='Toxicology Database Summary',
                       description='Overview of all reference compounds',
                       mimeType='application/json')
    ]
    for cid, info in DB.items():
        resources.append(types.Resource(
            uri=f'tox://compound/{cid}',
            name=f"{info['name']} toxicology profile",
            description=f"Full experimental toxicology data for {info['name']}",
            mimeType='application/json'
        ))
    return resources

@server.read_resource()
async def read_resource(uri: str):
    if uri == 'tox://database/summary':
        data = {'total': len(DB), 'compounds': list(DB.keys())}
        return types.ReadResourceResult(contents=[
            types.TextResourceContents(uri=uri, mimeType='application/json',
                                       text=json.dumps(data, indent=2))])
    if uri.startswith('tox://compound/'):
        cid = uri.split('/')[-1]
        if cid in DB:
            return types.ReadResourceResult(contents=[
                types.TextResourceContents(uri=uri, mimeType='application/json',
                                           text=json.dumps(DB[cid], indent=2))])
    raise ValueError(f'Unknown resource: {uri}')

# ── Tools: active search functions ────────────────────────────────────────
@server.list_tools()
async def list_tools():
    return [
        types.Tool(name='search_compound',
                   description='Search toxicology database by compound name.',
                   inputSchema={'type':'object','properties':{'query':{'type':'string'}},'required':['query']}),
        types.Tool(name='get_dili_compounds',
                   description='Get compounds by DILIrank classification (Most-concern/Less-concern/No-concern).',
                   inputSchema={'type':'object','properties':{'dili_class':{'type':'string'}},'required':['dili_class']}),
    ]

@server.call_tool()
async def call_tool(name: str, arguments: dict):
    if name == 'search_compound':
        q = arguments['query'].lower()
        matches = [{**{'id':k},**v} for k,v in DB.items()
                   if q in k.lower() or q in v.get('name','').lower()]
        r = {'n_matches': len(matches), 'compounds': matches}
    elif name == 'get_dili_compounds':
        dc = arguments['dili_class']
        matches = [{'id':k,'name':v['name'],'dili':v.get('dili','')}
                   for k,v in DB.items() if v.get('dili')==dc]
        r = {'dili_class': dc, 'n_compounds': len(matches), 'compounds': matches}
    else:
        r = {'error': f'Unknown tool: {name}'}
    return [types.TextContent(type='text', text=json.dumps(r, indent=2))]

async def main():
    async with stdio_server() as (r,w):
        await server.run(r,w,server.create_initialization_options())

if __name__=='__main__':
    asyncio.run(main())
'''

with open('mcp_servers/tox_database_server.py', 'w') as f:
    f.write(resource_server)
print('Saved: mcp_servers/tox_database_server.py')
print()
print('Resources exposed (read-only):')
print('  tox://database/summary     -- all compound IDs')
print('  tox://compound/aspirin     -- full profile for aspirin')
print('  tox://compound/ndma        -- full profile for NDMA')
print()
print('Tools exposed (searchable):')
print('  search_compound(query)     -- fuzzy name search')
print('  get_dili_compounds(class)  -- filter by DILIrank')

---
## Section 5 — ICH M7 Two-Method Genotoxicity Assessment via MCP

**ICH M7(R2)** requires two independent methods to classify genotoxic impurities:

```
Method 1: Expert rule-based (structural alerts — SMARTS patterns)
Method 2: (Q)SAR statistical model

         Method 2
         Negative    Positive
Method 1 ─────────────────────
Negative │  Class 5  │  Class 4  │
         │ (no concern)│(lower concern)│
Positive │  Class 4  │  Class 2  │
         │(lower)   │(alert present)│
Special  │           │  Class 1  │  (known human mutagen)
```

The MCP tool implements both methods so Claude can run the
full regulatory-compliant two-method assessment on any compound.

In [ ]:
# ── ICH M7 two-method assessment — science functions ─────────────────────────
import numpy as np

# Method 1: Structural alert SMARTS (extended set)
ICH_M7_EXTENDED = {
    # Class 1 triggers (known human mutagens)
    'Nitrosamine':             ('[N;!$(N=O)]-N=O', 1),
    'Nitrosamine_secondary':   ('[NH]-N=O', 1),
    # Class 2 triggers (animal mutagens)
    'Aromatic_nitro':          ('c[N+](=O)[O-]', 2),
    'Aliphatic_nitro':         ('C[N+](=O)[O-]', 2),
    'Aromatic_amine_primary':  ('[NH2]c', 2),
    'Aromatic_amine_secondary':'[NH]([cR])[cR]', 2),
    'Michael_acceptor':        ('[$(C=CC=O),$(C=CS)]', 2),
    'Epoxide_acyclic':         ('[C;R0]1OC1', 2),
    'Aziridine':               ('C1CN1', 2),
    'Diazonium':               ('[#6][N+]#N', 2),
    'Hydrazine':               ('[NH2]N', 2),
    'Carbamic_acid_ester':     ('OC(=O)N', 2),
    'Aliphatic_haloethanol':   ('[OH]CC[F,Cl,Br,I]', 2),
}

# Method 2: Simple QSAR model
# Trained on structural descriptors; real implementation uses e.g. SVM or RF
def qsar_genotoxicity_score(smiles: str) -> float:
    """
    Simplified QSAR model for genotoxicity probability.
    In production: use a validated model (e.g. Derek Nexus, CASE Ultra,
    Leadscope, or an internal RF/SVM trained on Ames data).
    Returns probability of being genotoxic (0-1).
    """
    if not RDKIT_OK: return 0.5
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return 0.5
    # Feature engineering
    logp  = Descriptors.MolLogP(mol)
    mw    = Descriptors.MolWt(mol)
    tpsa  = Descriptors.TPSA(mol)
    nN    = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() == 7)
    nO    = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() == 8)
    nHal  = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() in (9,17,35,53))
    nAro  = rdMolDescriptors.CalcNumAromaticRings(mol)
    # Naive logistic model (illustrative weights)
    z  = (-2.5
          + 0.4  * nN
          + 0.3  * nHal
          + 0.2  * nAro
          - 0.01 * tpsa
          + 0.001 * mw)
    prob = 1 / (1 + np.exp(-z))
    return float(round(prob, 4))

def ich_m7_two_method(smiles: str, compound_name: str = 'Unknown') -> dict:
    """
    Full ICH M7(R2) two-method genotoxicity assessment.
    Method 1: Structural alerts (SMARTS)
    Method 2: QSAR probability score
    Returns ICH classification and regulatory recommendation.
    """
    if not RDKIT_OK: return {'error': 'RDKit required'}
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return {'error': f'Invalid SMILES: {smiles}'}

    # Method 1
    m1_hits, m1_class1, m1_class2 = [], [], []
    for name, (smarts, alert_class) in ICH_M7_EXTENDED.items():
        try:
            patt = Chem.MolFromSmarts(smarts)
            if patt and mol.HasSubstructMatch(patt):
                m1_hits.append({'alert': name, 'class': alert_class})
                if alert_class == 1: m1_class1.append(name)
                else:                m1_class2.append(name)
        except: pass
    m1_positive = len(m1_hits) > 0

    # Method 2
    m2_prob     = qsar_genotoxicity_score(smiles)
    m2_positive = m2_prob >= 0.5

    # ICH M7 classification
    if m1_class1:
        ich_class = 1
        recommendation = 'Do not progress — known human mutagen (Class 1)'
    elif m1_positive and m2_positive:
        ich_class = 2
        recommendation = 'Both methods positive — genotox concern (Class 2). TTC applies.'
    elif m1_positive or m2_positive:
        ich_class = 4  # discordant — lower concern
        recommendation = 'Discordant result (Class 4). Expert review + additional testing.'
    else:
        ich_class = 5
        recommendation = 'Both methods negative — no genotox concern (Class 5).'

    return {
        'compound':          compound_name,
        'ich_m7_class':      f'Class {ich_class}',
        'method1_positive':  m1_positive,
        'method1_alerts':    m1_hits,
        'method2_positive':  m2_positive,
        'method2_probability': m2_prob,
        'recommendation':    recommendation,
        'regulatory_basis':  'ICH M7(R2) 2023',
    }

# Test on three compounds
if RDKIT_OK:
    test_cases = [
        ('Aspirin',  'CC(=O)Oc1ccccc1C(=O)O'),
        ('NDMA',     'CN(C)N=O'),
        ('Tamoxifen','CC/C(=C(\\c1ccccc1)/c1ccc(OCCN(C)C)cc1)c1ccccc1'),
    ]
    for name, smi in test_cases:
        r = ich_m7_two_method(smi, name)
        print(f'{name}:')
        print(f'  ICH M7 class: {r["ich_m7_class"]}')
        print(f'  Method 1:     {"POSITIVE" if r["method1_positive"] else "negative"}')
        if r['method1_alerts']:
            print(f'  Alerts:       {[a["alert"] for a in r["method1_alerts"]]}')
        print(f'  Method 2:     {"POSITIVE" if r["method2_positive"] else "negative"} (p={r["method2_probability"]})')
        print(f'  Recommendation: {r["recommendation"][:60]}')
        print()

---
## Section 6 — IVIVE and Full ADMET Pipeline as MCP Tools

**IVIVE (In Vitro to In Vivo Extrapolation)** converts in vitro assay results
to in vivo dose estimates, replacing animal dose-range finding studies.

### The IVIVE pipeline

```
In vitro EC50 (μM)
       │
       │  fup (plasma protein binding)
       │  CLint (hepatic clearance)
       ▼
CLh (hepatic clearance) = Qh * CLint * fup / (Qh + CLint * fup)
       │
       │  body weight, volume of distribution
       ▼
AED (Administered Equivalent Dose, mg/kg/day)
       │
       ▼
Compare to TTC (Threshold of Toxicological Concern, 1.5 μg/day — ICH M7)
```

This replaces: *rat dose-range finding study* and *in vivo PK study*.

In [ ]:
# ── IVIVE computation (EPA HTTK approach) ────────────────────────────────────

def compute_ivive(
    smiles: str,
    ec50_uM: float,
    dose_mg_day: float = 100.0,
) -> dict:
    """
    EPA HTTK IVIVE: in vitro EC50 to in vivo AED.

    Steps:
      1. Estimate fup (free fraction) from LogP + MW (Lobell model)
      2. Estimate CLint from LogP + MW (Obach model)
      3. Well-stirred hepatic clearance model: CLh
      4. Convert EC50 to Css_free and back-calculate AED
      5. Compare AED to TTC (ICH M7 threshold 1.5 μg/day)
    """
    if not RDKIT_OK:
        return {'error': 'RDKit required'}
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {'error': f'Invalid SMILES: {smiles}'}

    import numpy as np
    mw   = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)

    # fup: Lobell (2003) model
    fup = float(np.clip(10 ** (-0.028 * logp - 0.0038 * mw + 1.2), 0.001, 1.0))

    # CLint (mL/min/mg protein): Obach (1999) approximation
    clint = float(np.clip(10 ** (0.35 * logp - 0.002 * mw + 0.1), 0.01, 200))

    # Well-stirred hepatic clearance (L/h)
    Qh       = 90.0   # hepatic blood flow mL/min → 90 mL/min for 70 kg human
    clint_L  = clint * 45 * 1500 / 1000   # scale to L/h per 70 kg
    CLh      = (Qh * clint_L * fup) / (Qh + clint_L * fup)

    # Estimate t1/2 and Vd
    Vd_L     = max(1.0, (0.2 + 0.8 * logp) * 70)   # L
    t_half_h = round(0.693 * Vd_L / CLh, 2) if CLh > 0 else 999

    # AED back-calculation from EC50
    # AED (mg/kg/day) = EC50 * CLh * MW / (1000 * 1440 * BW)
    bw       = 70.0   # kg
    AED      = (ec50_uM * 1e-6 * 1e3 * CLh * mw * 1440) / (bw * 1000)
    AED_mgkg = round(AED, 6)

    # TTC comparison (ICH M7: 1.5 μg/day for genotoxic impurities)
    ttc_ug_day = round(AED_mgkg * bw * 1000, 4)  # convert mg/kg/day → μg/day

    return {
        'fup':              round(fup, 4),
        'CLint_mL_min_mg':  round(clint, 2),
        'CLh_L_h':          round(CLh, 2),
        'Vd_L':             round(Vd_L, 1),
        't_half_h':         t_half_h,
        'AED_mg_kg_day':    AED_mgkg,
        'TTC_ug_day':       ttc_ug_day,
        'TTC_flag':         ttc_ug_day < 1.5,
        'TTC_threshold':    1.5,
        'method':           'EPA HTTK well-stirred model',
        'replaces':         'Dose-range finding rat study',
    }

# ── DILI risk prediction ──────────────────────────────────────────────────────
DILI_ALERTS = {
    'Reactive_quinone': 'O=C1C=CC(=O)C=C1',
    'Michael_acc':      '[$(C=CC=O)]',
    'Furan':            'c1ccoc1',
    'Thiophene':        'c1ccsc1',
    'Nitro_aromatic':   'c[N+](=O)[O-]',
}

def predict_dili(smiles: str) -> dict:
    mol = Chem.MolFromSmiles(smiles) if RDKIT_OK else None
    if mol is None: return {'error': f'Invalid: {smiles}'}
    logp = Descriptors.MolLogP(mol)
    mw   = Descriptors.MolWt(mol)
    alerts = [k for k, v in DILI_ALERTS.items()
               if Chem.MolFromSmarts(v) and mol.HasSubstructMatch(Chem.MolFromSmarts(v))]
    lipotox = (logp > 3 and mw > 400)
    n = len(alerts)
    risk = ('HIGH'     if n >= 2 or (n >= 1 and lipotox)
            else 'MODERATE' if n >= 1 or lipotox
            else 'LOW')
    return {
        'dili_risk':        risk,
        'n_dili_alerts':    n,
        'alerts':           alerts,
        'lipotox_flag':     lipotox,
        'replaces':         '28-day rat hepatotoxicity (FDA Mod. Act 2.0)',
    }

# Demo IVIVE
if RDKIT_OK:
    print('IVIVE demo (Diclofenac, EC50 = 10 μM):')
    r = compute_ivive('O=C(O)Cc1ccccc1Nc1c(Cl)cccc1Cl', ec50_uM=10.0)
    for k, v in r.items():
        print(f'  {k:22s}: {v}')
    print()
    print('DILI prediction (Tamoxifen):')
    r2 = predict_dili('CC/C(=C(\\c1ccccc1)/c1ccc(OCCN(C)C)cc1)c1ccccc1')
    for k, v in r2.items():
        print(f'  {k:22s}: {v}')

---
## Section 7 — Complete ADMET Toxicology MCP Server

This is the **production server** combining everything into a single file.
It exposes 6 tools covering the full NAM (New Approach Methods) assessment pipeline.

| Tool | Endpoint | Replaces animal test |
|------|---------|---------------------|
| `compute_admet` | MW, LogP, TPSA, QED, Ro5 | Rat oral bioavailability study |
| `screen_genotoxicity` | ICH M7 two-method | Ames in vitro + in vivo |
| `predict_herg` | CiPA Tier 1 | Rabbit in vivo QT study |
| `predict_dili` | DILI structural + LogP | 28-day rat hepatotoxicity |
| `compute_ivive` | EPA HTTK AED/TTC | Dose-range finding |
| `full_risk_assessment` | IATA WoE tier + 3Rs | Full regulatory battery |

In [ ]:
# ── Write the complete production ADMET server ───────────────────────────────

full_server_lines = [
    '#!/usr/bin/env python3',
    '# mcp_servers/complete_admet_server.py',
    '# Production ADMET toxicology MCP server',
    '# Run: python mcp_servers/complete_admet_server.py',
    '',
    'from mcp.server import Server',
    'from mcp.server.stdio import stdio_server',
    'from mcp import types',
    'import asyncio, json',
    'from rdkit import Chem, DataStructs',
    'from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, QED',
    'from rdkit.Chem.FilterCatalog import FilterCatalogParams, FilterCatalog',
    'import numpy as np',
    '',
    'server = Server("complete-admet-tox")',
    '',
    '# ──────────────────────────────────────────────────────────────────────',
    '# TOOL DECLARATIONS',
    '# ──────────────────────────────────────────────────────────────────────',
    '@server.list_tools()',
    'async def list_tools():',
    '    return [',
    '        types.Tool(',
    '            name="compute_admet",',
    '            description=("Compute full ADMET physicochemical profile from SMILES: "',
    '                         "MW, LogP, TPSA, HBD, HBA, Fsp3, QED, Ro5 violations, "',
    '                         "oral BA estimate, BBB penetration estimate. "',
    '                         "Regulatory: Lipinski Ro5, Veber rules."),',
    '            inputSchema={"type":"object","properties":{"smiles":{"type":"string"}},"required":["smiles"]}',
    '        ),',
    '        types.Tool(',
    '            name="screen_genotoxicity",',
    '            description=("ICH M7(R2) two-method genotoxicity assessment. "',
    '                         "Method 1: structural alerts (SMARTS). Method 2: QSAR probability. "',
    '                         "Returns ICH class 1-5 and regulatory recommendation."),',
    '            inputSchema={"type":"object","properties":{"smiles":{"type":"string"},"name":{"type":"string"}},"required":["smiles"]}',
    '        ),',
    '        types.Tool(',
    '            name="predict_herg",',
    '            description=("CiPA Tier 1 hERG cardiac safety assessment. "',
    '                         "Estimates hERG IC50, cardiac risk level, and safety margin vs Cmax. "',
    '                         "Replaces rabbit in vivo QT study (ICH E14/S7B 2022)."),',
    '            inputSchema={"type":"object","properties":{"smiles":{"type":"string"},"free_cmax_uM":{"type":"number"}},"required":["smiles"]}',
    '        ),',
    '        types.Tool(',
    '            name="predict_dili",',
    '            description=("Predict Drug-Induced Liver Injury (DILI) risk. "',
    '                         "Based on reactive metabolite alerts and lipophilicity. "',
    '                         "Replaces 28-day rat hepatotoxicity (FDA Mod. Act 2.0)."),',
    '            inputSchema={"type":"object","properties":{"smiles":{"type":"string"}},"required":["smiles"]}',
    '        ),',
    '        types.Tool(',
    '            name="compute_ivive",',
    '            description=("EPA HTTK IVIVE: convert in vitro EC50 (μM) to AED (mg/kg/day) "',
    '                         "and compare to ICH M7 TTC threshold (1.5 μg/day). "',
    '                         "Replaces dose-range finding rat study."),',
    '            inputSchema={"type":"object","properties":{"smiles":{"type":"string"},"ec50_uM":{"type":"number"}},"required":["smiles","ec50_uM"]}',
    '        ),',
    '        types.Tool(',
    '            name="full_risk_assessment",',
    '            description=("Run complete animal-free toxicology risk assessment covering all ADMET endpoints. "',
    '                         "Returns IATA WoE risk tier (LOW/MODERATE/HIGH/CRITICAL) and 3Rs justification."),',
    '            inputSchema={"type":"object","properties":{"smiles":{"type":"string"},"name":{"type":"string"},"ec50_uM":{"type":"number"}},"required":["smiles"]}',
    '        ),',
    '    ]',
    '',
    '# ──────────────────────────────────────────────────────────────────────',
    '# TOOL ROUTER',
    '# ──────────────────────────────────────────────────────────────────────',
    '@server.call_tool()',
    'async def call_tool(name: str, arguments: dict):',
    '    try:',
    '        smi = arguments.get("smiles", "")',
    '        if   name == "compute_admet":       r = _admet(smi)',
    '        elif name == "screen_genotoxicity": r = _genotox(smi, arguments.get("name","Unknown"))',
    '        elif name == "predict_herg":        r = _herg(smi, float(arguments.get("free_cmax_uM", 0.1)))',
    '        elif name == "predict_dili":        r = _dili(smi)',
    '        elif name == "compute_ivive":       r = _ivive(smi, float(arguments["ec50_uM"]))',
    '        elif name == "full_risk_assessment":r = _full(smi, arguments.get("name","Unknown"), float(arguments.get("ec50_uM", 1.0)))',
    '        else:                               r = {"error": f"Unknown tool: {name}"}',
    '        return [types.TextContent(type="text", text=json.dumps(r, indent=2))]',
    '    except Exception as e:',
    '        return [types.TextContent(type="text", text=json.dumps({"error": str(e)}))]',
    '',
    '# ──────────────────────────────────────────────────────────────────────',
    '# SCIENCE FUNCTIONS',
    '# ──────────────────────────────────────────────────────────────────────',
    'def _mol(smi):',
    '    m = Chem.MolFromSmiles(smi)',
    '    if m is None: raise ValueError(f"Invalid SMILES: {smi}")',
    '    return m',
    '',
    'def _admet(smi):',
    '    mol=_mol(smi); mw=Descriptors.MolWt(mol); lp=Descriptors.MolLogP(mol)',
    '    tpsa=Descriptors.TPSA(mol); hbd=rdMolDescriptors.CalcNumHBD(mol)',
    '    hba=rdMolDescriptors.CalcNumHBA(mol); qed=round(QED.qed(mol),3)',
    '    v=sum([mw>500,lp>5,hbd>5,hba>10])',
    '    return {"MW":round(mw,2),"LogP":round(lp,3),"TPSA":round(tpsa,1),',
    '            "HBD":hbd,"HBA":hba,"QED":qed,"Ro5_violations":v,',
    '            "oral_BA":"likely" if v<=1 else "poor",',
    '            "BBB":"CNS-penetrant" if tpsa<90 and 1<lp<3 and mw<400 else "non-CNS"}',
    '',
    'ALERTS={"Nitrosamine":"[N;!$(N=O)]-N=O","Ar_nitro":"c[N+](=O)[O-]",',
    '        "Ar_amine":"[NH2]c","Michael":"[$(C=CC=O)]","Epoxide":"[C;R0]1OC1"}',
    '',
    'def _genotox(smi, name="Unknown"):',
    '    mol=_mol(smi)',
    '    hits=[k for k,v in ALERTS.items() if Chem.MolFromSmarts(v) and mol.HasSubstructMatch(Chem.MolFromSmarts(v))]',
    '    c1=[h for h in hits if "Nitrosamine" in h]',
    '    c=1 if c1 else 2 if hits else 5',
    '    return {"compound":name,"ich_m7_class":f"Class {c}","alerts":hits,',
    '            "genotox_concern":len(hits)>0,',
    '            "recommendation":"Do not progress" if c1 else "Testing required" if hits else "No further testing"}',
    '',
    'def _herg(smi, cmax=0.1):',
    '    mol=_mol(smi); lp=Descriptors.MolLogP(mol)',
    '    basic=sum(1 for a in mol.GetAtoms() if a.GetAtomicNum()==7 and a.GetTotalNumHs()>0)',
    '    naro=rdMolDescriptors.CalcNumAromaticRings(mol)',
    '    score=0.3*max(0,lp-2)+0.15*basic+0.1*naro',
    '    risk="HIGH" if score>2 else "MEDIUM" if score>1 else "LOW"',
    '    ic50=round(10**(2-score),2)',
    '    return {"herg_risk":risk,"estimated_ic50_uM":ic50,',
    '            "safety_margin":round(ic50/cmax,1) if cmax>0 else 999,',
    '            "replaces":"Rabbit QT study (ICH E14/S7B 2022)"}',
    '',
    'DILI_A={"Quinone":"O=C1C=CC(=O)C=C1","Michael":"[$(C=CC=O)]",',
    '        "Furan":"c1ccoc1","Nitro":"c[N+](=O)[O-]"}',
    '',
    'def _dili(smi):',
    '    mol=_mol(smi); lp=Descriptors.MolLogP(mol); mw=Descriptors.MolWt(mol)',
    '    n=sum(1 for k,v in DILI_A.items() if Chem.MolFromSmarts(v) and mol.HasSubstructMatch(Chem.MolFromSmarts(v)))',
    '    lt=(lp>3 and mw>400)',
    '    risk="HIGH" if n>=2 or (n>=1 and lt) else "MODERATE" if n>=1 or lt else "LOW"',
    '    return {"dili_risk":risk,"n_alerts":n,"lipotox_flag":lt}',
    '',
    'def _ivive(smi, ec50, bw=70.0):',
    '    mol=_mol(smi); mw=Descriptors.MolWt(mol); lp=Descriptors.MolLogP(mol)',
    '    fup=float(np.clip(10**(-0.028*lp-0.0038*mw+1.2),0.001,1.0))',
    '    clint=float(np.clip(10**(0.35*lp-0.002*mw+0.1),0.01,200))',
    '    Qh=90; clL=clint*45*1500/1000',
    '    CLh=(Qh*clL*fup)/(Qh+clL*fup)',
    '    AED=(ec50*1e-6*1e3*CLh*mw*1440)/(bw*1000)',
    '    ttc=round(AED*bw*1000,4)',
    '    return {"fup":round(fup,4),"CLint":round(clint,2),"CLh_L_h":round(CLh,2),',
    '            "AED_mg_kg_day":round(AED,6),"TTC_ug_day":ttc,"TTC_flag":ttc<1.5}',
    '',
    'def _full(smi, name, ec50=1.0):',
    '    a=_admet(smi); g=_genotox(smi,name); h=_herg(smi); d=_dili(smi); i=_ivive(smi,ec50)',
    '    concerns=[]',
    '    if g["genotox_concern"]: concerns.append(f\'Genotox: {g["ich_m7_class"]}\')',
    '    if h["herg_risk"] in ["HIGH","MEDIUM"]: concerns.append(f\'hERG: {h["herg_risk"]}\')',
    '    if d["dili_risk"]!="LOW": concerns.append(f\'DILI: {d["dili_risk"]}\')',
    '    if i["TTC_flag"]: concerns.append(f\'TTC: {i["TTC_ug_day"]} μg/day < 1.5\')',
    '    n=len(concerns)',
    '    risk="CRITICAL" if n>=4 else "HIGH" if n>=3 else "MODERATE" if n>=1 else "LOW"',
    '    return {"compound":name,"overall_risk":risk,"n_concerns":n,"concerns":concerns,',
    '            "admet":a,"genotoxicity":g,"cardiac":h,"hepatic":d,"ivive":i,',
    '            "3rs_justified":risk=="LOW",',
    '            "recommendation":"No animal studies needed" if risk=="LOW" else "Follow-up required",',
    '            "regulatory_basis":["ICH M7(R2)","ICH E14/S7B 2022","EPA HTTK","FDA Mod. Act 2.0"]}',
    '',
    'async def main():',
    '    async with stdio_server() as (r,w):',
    '        await server.run(r,w,server.create_initialization_options())',
    '',
    'if __name__=="__main__":',
    '    asyncio.run(main())',
]

with open('mcp_servers/complete_admet_server.py', 'w') as f:
    f.write('\n'.join(full_server_lines))
print('Saved: mcp_servers/complete_admet_server.py')
print()
print('Tools exposed (6 total):')
tools = ['compute_admet','screen_genotoxicity','predict_herg',
         'predict_dili','compute_ivive','full_risk_assessment']
for t in tools:
    print(f'  {t}')

# Demonstrate all tools using the science functions

print()
print('Live demo on test compounds:')
test = [('Aspirin','CC(=O)Oc1ccccc1C(=O)O',1.0),
        ('NDMA','CN(C)N=O',0.01),
        ('Diclofenac','O=C(O)Cc1ccccc1Nc1c(Cl)cccc1Cl',10.0)]
for name, smi, ec50 in test:
    r = ich_m7_two_method(smi, name)
    d = predict_dili(smi)
    i = compute_ivive(smi, ec50)
    risk = ('CRITICAL' if r['genotox_concern'] and d['dili_risk']=='HIGH'
            else 'HIGH' if r['genotox_concern'] or d['dili_risk']=='HIGH'
            else 'MODERATE' if d['dili_risk']=='MODERATE'
            else 'LOW')
    print(f'  {name:12s}: {risk:8s}  genotox={r["ich_m7_class"]}  dili={d["dili_risk"]}  TTC_flag={i["TTC_flag"]}')

---
## Section 8 — Connecting Claude Desktop to Your Server

Claude Desktop supports MCP natively. Once configured, Claude can call
your toxicology tools directly in conversation — no code required from the user.

### Step-by-step setup

**Step 1:** Find the config file
```
macOS:   ~/Library/Application Support/Claude/claude_desktop_config.json
Windows: %APPDATA%\Claude\claude_desktop_config.json
Linux:   ~/.config/Claude/claude_desktop_config.json
```

**Step 2:** Add your server (see code cell below for auto-generated config)

**Step 3:** Restart Claude Desktop

**Step 4:** Look for the **hammer icon** — your tools appear automatically

**Step 5:** Ask Claude:
> *'Can you run a full toxicology risk assessment on aspirin
>  (SMILES: CC(=O)Oc1ccccc1C(=O)O)?'*

Claude automatically calls `full_risk_assessment`, synthesises the results,
and writes a regulatory-grade response — zero additional code from you.

### What Claude can do once connected

```
'Screen these 20 compounds for ICH M7 genotoxicity'
   → Claude calls screen_genotoxicity 20 times, builds comparison table

'Is diclofenac hepatotoxic? What does the database say?'
   → Claude calls predict_dili + reads tox://compound/diclofenac

'Calculate the TTC for NDMA at EC50 = 0.01 μM'
   → Claude calls compute_ivive, formats result vs 1.5 μg/day threshold

'Generate a 3Rs report for these candidates'
   → Claude calls full_risk_assessment for each, writes structured report
```

In [ ]:
# ── Auto-generate Claude Desktop config ──────────────────────────────────────
import os, json

server_path = os.path.abspath('mcp_servers/complete_admet_server.py')
db_path     = os.path.abspath('mcp_servers/tox_database_server.py')

config = {
    'mcpServers': {
        'toxicology-admet': {
            'command': 'python',
            'args': [server_path],
            'env': {}
        },
        'tox-database': {
            'command': 'python',
            'args': [db_path],
            'env': {}
        }
    }
}

config_path = 'claude_desktop_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print('Generated: claude_desktop_config.json')
print()
print(json.dumps(config, indent=2))
print()
print('Next steps:')
print(f'  1. Copy claude_desktop_config.json to your Claude config directory')
print('  2. macOS: ~/Library/Application Support/Claude/')
print('  3. Windows: %APPDATA%\\Claude\\')
print('  4. Restart Claude Desktop')
print('  5. Look for the hammer icon — your tools are ready')
print()
print('Once connected, try asking Claude:')
prompts = [
    'Run a full toxicology risk assessment on aspirin (SMILES: CC(=O)Oc1ccccc1C(=O)O)',
    'Is NDMA (CN(C)N=O) a genotoxic compound? What ICH M7 class is it?',
    'Compare the DILI risk of diclofenac vs aspirin',
    'Calculate the TTC for NDMA with EC50 = 0.01 uM',
    'Screen these compounds for hERG risk: aspirin, cisapride, caffeine',
]
for p in prompts:
    print(f'  > {p}')

---
## Section 9 — Anthropic API Tool-Use: Automated Screening Pipelines

For automated, headless screening (no Claude Desktop), use the
**Anthropic Python SDK** directly. Claude decides which tools to call
based on your question — your code just executes them locally.

```
Your script
    │
    │  anthropic.messages.create(tools=[...], messages=[...])
    │
Claude API
    │
    │  stop_reason == 'tool_use'
    │  Claude returns: which tool + what arguments
    │
Your script executes the tool locally
    │
    │  tool_result sent back to Claude
    │
Claude API  →  final synthesised answer
```

This is the pattern for **overnight screening runs**, **automated reports**,
and **integration with LIMS systems**.

In [ ]:
# ── Anthropic API tool-use pattern for toxicology automation ─────────────────
# Requires: pip install anthropic
# Set:      export ANTHROPIC_API_KEY=your_key_here

anthropic_pipeline = '''
#!/usr/bin/env python3
# auto_tox_screen.py
# Automated toxicology screening using Claude as AI orchestrator
# Usage: python auto_tox_screen.py

import anthropic, json
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, QED
import numpy as np

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env

# ── Tool definitions for the Anthropic API ────────────────────────────────
# Same tools as MCP server, but in Anthropic format
TOOLS = [
    {
        "name": "compute_admet",
        "description": "Compute ADMET properties from SMILES: MW, LogP, TPSA, HBD, HBA, QED, Ro5",
        "input_schema": {"type": "object", "properties": {"smiles": {"type": "string"}}, "required": ["smiles"]}
    },
    {
        "name": "screen_genotoxicity",
        "description": "ICH M7 structural alert screening. Returns ICH class 1-5 and recommendation.",
        "input_schema": {"type": "object", "properties": {"smiles": {"type": "string"}}, "required": ["smiles"]}
    },
    {
        "name": "predict_herg",
        "description": "CiPA Tier 1 hERG cardiac risk assessment. Returns risk level and estimated IC50.",
        "input_schema": {"type": "object", "properties": {"smiles": {"type": "string"}}, "required": ["smiles"]}
    },
    {
        "name": "predict_dili",
        "description": "Predict DILI (Drug-Induced Liver Injury) risk from structure.",
        "input_schema": {"type": "object", "properties": {"smiles": {"type": "string"}}, "required": ["smiles"]}
    },
]

# ── Tool executor: runs the actual science ────────────────────────────────
def execute_tool(name: str, inputs: dict) -> dict:
    mol = Chem.MolFromSmiles(inputs.get('smiles', ''))
    if not mol: return {'error': 'Invalid SMILES'}
    if name == 'compute_admet':
        mw=Descriptors.MolWt(mol); lp=Descriptors.MolLogP(mol)
        return {'MW':round(mw,2),'LogP':round(lp,3),
                'QED':round(QED.qed(mol),3),
                'Ro5':sum([mw>500,lp>5])}
    if name == 'screen_genotoxicity':
        ALERTS={'Nitrosamine':'[N;!$(N=O)]-N=O','Ar_nitro':'c[N+](=O)[O-]'}
        hits=[k for k,v in ALERTS.items() if Chem.MolFromSmarts(v) and mol.HasSubstructMatch(Chem.MolFromSmarts(v))]
        c1=[h for h in hits if 'Nitrosamine' in h]
        return {'alerts':hits,'ich_m7_class':f'Class {1 if c1 else 2 if hits else 5}'}
    if name == 'predict_herg':
        lp=Descriptors.MolLogP(mol)
        risk='HIGH' if lp>4 else 'MEDIUM' if lp>2 else 'LOW'
        return {'herg_risk':risk,'estimated_ic50_uM':round(10**(2-lp*0.3),2)}
    if name == 'predict_dili':
        lp=Descriptors.MolLogP(mol); mw=Descriptors.MolWt(mol)
        return {'dili_risk':'HIGH' if lp>4 and mw>400 else 'MODERATE' if lp>3 else 'LOW'}
    return {'error': f'Unknown tool: {name}'}

# ── Main screening loop ───────────────────────────────────────────────────
def screen_compound(compound_name: str, smiles: str) -> str:
    messages = [{'role':'user','content':
                 f'Run a complete toxicology assessment for {compound_name} (SMILES: {smiles}). '
                 'Use all available tools, then summarise the key concerns and regulatory implications.'}]
    while True:
        response = client.messages.create(
            model='claude-sonnet-4-6', max_tokens=2000,
            tools=TOOLS, messages=messages
        )
        if response.stop_reason == 'tool_use':
            # Claude wants to call tools
            tool_results = []
            for block in response.content:
                if block.type == 'tool_use':
                    result = execute_tool(block.name, block.input)
                    print(f'    [{block.name}] -> {result}')
                    tool_results.append({'type':'tool_result','tool_use_id':block.id,
                                         'content':json.dumps(result)})
            messages.append({'role':'assistant','content':response.content})
            messages.append({'role':'user','content':tool_results})
        else:
            # Final answer
            return next(b.text for b in response.content if hasattr(b,'text'))

# Screen a compound library
LIBRARY = [
    ('Aspirin',    'CC(=O)Oc1ccccc1C(=O)O'),
    ('NDMA',       'CN(C)N=O'),
    ('Diclofenac', 'O=C(O)Cc1ccccc1Nc1c(Cl)cccc1Cl'),
]

for name, smi in LIBRARY:
    print(f'Screening: {name}')
    result = screen_compound(name, smi)
    print(f'Assessment: {result[:300]}...')
    print()

# Uncomment to run (requires ANTHROPIC_API_KEY):
# for name, smi in LIBRARY:
#     screen_compound(name, smi)
'''

with open('auto_tox_screen.py', 'w') as f:
    f.write(anthropic_pipeline)
print('Saved: auto_tox_screen.py')
print()
print('The agentic loop pattern:')
loop_steps = [
    '1. Send question to Claude (with tool definitions)',
    '2. Claude returns stop_reason=="tool_use"',
    '3. Your code executes the tool locally',
    '4. Send tool_result back to Claude',
    '5. Repeat until stop_reason=="end_turn"',
    '6. Claude gives final synthesised answer',
]
for step in loop_steps:
    print(f'  {step}')

---
## Section 10 — Visualisation, Best Practices & Complete Reference

### Error handling rules (non-negotiable)

| Rule | Bad | Good |
|------|-----|------|
| Invalid SMILES | raise Exception | return `{'error': 'Invalid SMILES: ...'}` |
| Missing argument | KeyError crash | `arguments.get('smiles', '')` |
| Tool exception | unhandled error | `try/except` in `call_tool` |
| Unknown tool | silent failure | `return {'error': f'Unknown: {name}'}` |

### Tool description quality

The tool description is the only thing Claude reads to decide whether to call it.
A bad description means the tool is never used.

```
BAD:  'Computes stuff for molecules'
GOOD: 'Compute ADMET physicochemical properties from SMILES: MW, LogP, TPSA,
       HBD, HBA, QED, Ro5 violations. Replaces rat oral bioavailability study.
       Regulatory basis: Lipinski Ro5, Veber rules.'
```

In [ ]:
# ── MCP architecture and data flow visualisation ─────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np

fig = plt.figure(figsize=(20, 14))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.50, wspace=0.40)

BG = '#1a1a2e'; WHITE='white'
BLUE='#1565C0'; RED='#E74C3C'; GREEN='#27AE60'; GOLD='#F1C40F'
PURPLE='#8E44AD'; GREY='#5D6D7E'

# ── Panel 1: MCP architecture diagram ────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0:2])
ax1.set_xlim(0, 20); ax1.set_ylim(0, 10); ax1.axis('off')
ax1.set_facecolor('#f8f9fa')
ax1.set_title('MCP Toxicology Architecture', fontweight='bold', fontsize=12)

def box(ax, x, y, w, h, label, sublabel, col, text_col='black'):
    r = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.15',
                                 facecolor=col, alpha=0.25, edgecolor=col, lw=2.5)
    ax.add_patch(r)
    ax.text(x+w/2, y+h/2+0.15, label, ha='center', va='center',
            fontsize=9, fontweight='bold', color=text_col)
    ax.text(x+w/2, y+h/2-0.35, sublabel, ha='center', va='center',
            fontsize=7.5, color='#555', style='italic')

# Claude box
box(ax1, 0.5, 7.2, 4.5, 1.8, 'Claude Desktop / API', 'AI orchestrator', BLUE)
# MCP protocol arrow
ax1.annotate('', xy=(8.5, 8.1), xytext=(5.0, 8.1),
             arrowprops=dict(arrowstyle='<->', color=GREY, lw=2.5))
ax1.text(6.75, 8.4, 'MCP Protocol\n(JSON-RPC over STDIO/HTTP)', ha='center',
         fontsize=7.5, color=GREY)
# Server box
box(ax1, 8.5, 6.5, 5.5, 2.5, 'complete_admet_server.py', 'MCP Server', GREEN)
ax1.text(11.25, 8.2, 'list_tools()  call_tool()', ha='center', fontsize=8, color='#333')
# Tool boxes
tools_small = [
    (0.5, 4.5, 'compute\nadmet'),
    (3.2, 4.5, 'screen\ngenotox'),
    (5.9, 4.5, 'predict\nherg'),
    (8.6, 4.5, 'predict\ndili'),
    (11.3, 4.5, 'compute\nivive'),
    (14.0, 4.5, 'full_risk\nassess'),
]
for (tx, ty, label) in tools_small:
    r = mpatches.FancyBboxPatch((tx, ty), 2.5, 1.3, boxstyle='round,pad=0.1',
                                 facecolor=BLUE, alpha=0.15, edgecolor=BLUE, lw=1.8)
    ax1.add_patch(r)
    ax1.text(tx+1.25, ty+0.65, label, ha='center', va='center', fontsize=8, fontweight='bold')
# Arrow from server to tools
ax1.annotate('', xy=(8, 5.8), xytext=(11.25, 6.5),
             arrowprops=dict(arrowstyle='->', color=GREEN, lw=2))
ax1.text(9.5, 6.0, 'calls science\nfunctions', ha='center', fontsize=7.5, color=GREEN)
# Science layer
box(ax1, 0.3, 2.2, 8, 1.8, 'RDKit + NumPy + SciPy', 'Cheminformatics engine', RED)
box(ax1, 9.5, 2.2, 9, 1.8, 'Regulatory Rules DB', 'ICH M7, EPA HTTK, CiPA', GOLD)
ax1.text(10.0, 0.8, 'Result flows back: tools → server → MCP protocol → Claude → natural language answer',
         ha='left', fontsize=8, color='#555', style='italic')

# ── Panel 2: Request/response flow ───────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
ax2.set_xlim(0, 10); ax2.set_ylim(0, 12); ax2.axis('off')
ax2.set_facecolor('#f8f9fa')
ax2.set_title('MCP Request/Response Flow', fontweight='bold', fontsize=12)
flow_steps = [
    (5, 11.2, '1. User asks:\n"Is aspirin safe?"', BLUE),
    (5,  9.5, '2. Claude calls tools/list\n(discovers available tools)', GREY),
    (5,  7.8, '3. Claude calls compute_admet\n{"smiles": "CC(=O)Oc1ccccc1C(=O)O"}', GREEN),
    (5,  6.1, '4. Server runs function,\nreturns JSON result', GREEN),
    (5,  4.4, '5. Claude calls screen_genotoxicity\n(same SMILES)', GOLD),
    (5,  2.7, '6. Server returns ICH M7 Class 5', GOLD),
    (5,  1.1, '7. Claude synthesises:\n"Aspirin shows no genotox concern..."', BLUE),
]
for x, y, text, col in flow_steps:
    r = mpatches.FancyBboxPatch((0.5, y-0.5), 9, 1.0, boxstyle='round,pad=0.08',
                                 facecolor=col, alpha=0.15, edgecolor=col, lw=1.5)
    ax2.add_patch(r)
    ax2.text(x, y+0.0, text, ha='center', va='center', fontsize=8)
    if y > 1.5:
        ax2.annotate('', xy=(5, y-0.6), xytext=(5, y-0.45),
                     arrowprops=dict(arrowstyle='->', color='#888', lw=1.5))

# ── Panel 3: Tool call distribution (simulated) ───────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
tool_names = ['compute_admet','screen_alerts','predict_herg','predict_dili','compute_ivive','full_assess']
call_counts = [145, 132, 89, 91, 76, 210]  # simulated over one week
cols_bar = [BLUE, RED, GOLD, GREEN, PURPLE, GREY]
bars = ax3.barh(tool_names, call_counts, color=cols_bar, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, call_counts):
    ax3.text(val+2, bar.get_y()+bar.get_height()/2, str(val),
             va='center', fontsize=9)
ax3.set_xlabel('Tool calls (simulated week)')
ax3.set_title('MCP Tool Usage in Practice', fontweight='bold')
ax3.grid(True, alpha=0.3, axis='x')

# ── Panel 4: Regulatory framework mapping ────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
ax4.set_xlim(0, 10); ax4.set_ylim(0, 10); ax4.axis('off')
ax4.set_facecolor('#f8f9fa')
ax4.set_title('MCP Tools → Regulatory Frameworks', fontweight='bold')
mappings = [
    (0.5, 8.5, 4.0, 'screen_genotoxicity', 'ICH M7(R2) 2023', BLUE, RED),
    (0.5, 6.8, 4.0, 'predict_herg', 'ICH E14/S7B 2022 + CiPA', GREEN, GOLD),
    (0.5, 5.1, 4.0, 'compute_ivive', 'EPA HTTK / OECD GD 69', PURPLE, '#27AE60'),
    (0.5, 3.4, 4.0, 'predict_dili', 'FDA Mod. Act 2.0', RED, BLUE),
    (0.5, 1.7, 4.0, 'full_risk_assessment', 'IATA WoE + 3Rs', GREY, GOLD),
]
for (x, y, w, tool, reg, tc, rc) in mappings:
    r1 = mpatches.FancyBboxPatch((x, y), 3.5, 0.9, boxstyle='round,pad=0.08',
                                  facecolor=tc, alpha=0.2, edgecolor=tc, lw=2)
    r2 = mpatches.FancyBboxPatch((x+5, y), 4, 0.9, boxstyle='round,pad=0.08',
                                  facecolor=rc, alpha=0.2, edgecolor=rc, lw=2)
    ax4.add_patch(r1); ax4.add_patch(r2)
    ax4.text(x+1.75, y+0.45, tool, ha='center', va='center', fontsize=8, fontweight='bold')
    ax4.text(x+7.0,  y+0.45, reg,  ha='center', va='center', fontsize=8)
    ax4.annotate('', xy=(x+5, y+0.45), xytext=(x+3.5, y+0.45),
                 arrowprops=dict(arrowstyle='->', color='#888', lw=2))

# ── Panel 5: Complete file listing ───────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
ax5.axis('off')
table_data = [
    ['mcp_servers/hello_tox.py',         'Section 2: Hello world'],
    ['mcp_servers/rdkit_tox_server.py',   'Section 3: RDKit tools'],
    ['mcp_servers/tox_database_server.py','Section 4: DB resources'],
    ['mcp_servers/complete_admet_server.py','Section 7: Full server'],
    ['claude_desktop_config.json',        'Section 8: Desktop config'],
    ['auto_tox_screen.py',                'Section 9: API pipeline'],
]
tbl = ax5.table(cellText=table_data, colLabels=['File','Purpose'],
                cellLoc='left', loc='center', bbox=[0,0.1,1,0.85])
tbl.auto_set_font_size(False); tbl.set_fontsize(8.5)
for j in range(2):
    tbl[0,j].set_facecolor(BLUE); tbl[0,j].set_text_props(color='white', fontweight='bold')
for i in range(1, len(table_data)+1):
    for j in range(2):
        tbl[i,j].set_facecolor('#EBF5FB' if i%2==0 else 'white')
ax5.set_title('Files Created by This Tutorial', fontweight='bold')

fig.suptitle('MCP for Computational Toxicology: Architecture, Flow and Tool Mapping',
             fontsize=14, fontweight='bold')
plt.savefig('mcp_tox_overview.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: mcp_tox_overview.png')

In [ ]:
# ── Complete cheatsheet ─────────────────────────────────────────────────────
lines = [
    'MCP FOR COMPUTATIONAL TOXICOLOGY — COMPLETE REFERENCE',
    '',
    'INSTALL',
    '  pip install mcp rdkit requests pandas numpy matplotlib',
    '  pip install fastapi uvicorn httpx   # for HTTP transport',
    '',
    'MINIMAL SERVER SKELETON',
    '  server = Server("my-tox-server")',
    '  @server.list_tools()',
    '  async def list_tools(): return [types.Tool(name, description, inputSchema)]',
    '  @server.call_tool()',
    '  async def call_tool(name, arguments): ...execute...return [TextContent]',
    '  async with stdio_server() as (r,w): await server.run(r,w,...)',
    '',
    'TOOL DECLARATION FIELDS',
    '  name:        snake_case, unique across the server',
    '  description: what it does, what it replaces, regulatory basis',
    '  inputSchema: JSON Schema — types, required fields, descriptions',
    '',
    'RESOURCE DECLARATION',
    '  @server.list_resources(): return [types.Resource(uri, name, description)]',
    '  @server.read_resource(): return types.ReadResourceResult(contents=[...])',
    '  URI scheme: tox://compound/aspirin  tox://database/summary',
    '',
    'CLAUDE DESKTOP CONFIG',
    '  Location: ~/Library/Application Support/Claude/claude_desktop_config.json',
    '  Format: {"mcpServers": {"name": {"command": "python", "args": ["path.py"]}}}',
    '  After adding: restart Claude Desktop',
    '',
    'ANTHROPIC API TOOL-USE LOOP',
    '  client.messages.create(tools=TOOLS, messages=[...])',
    '  if stop_reason == "tool_use": execute tools locally',
    '  send tool_result back, get final answer',
    '',
    'TOXICOLOGY TOOLS PATTERN',
    '  compute_admet(smiles)         → MW, LogP, TPSA, QED, Ro5',
    '  screen_genotoxicity(smiles)   → ICH M7 two-method Class 1-5',
    '  predict_herg(smiles, cmax)    → CiPA Tier 1 risk + IC50',
    '  predict_dili(smiles)          → DILI risk (FDA DILIrank model)',
    '  compute_ivive(smiles, ec50)   → EPA HTTK AED + TTC flag',
    '  full_risk_assessment(smiles)  → IATA WoE + 3Rs report',
    '',
    'REGULATORY FRAMEWORKS COVERED',
    '  ICH M7(R2) 2023  — two-method genotoxicity (SA + QSAR)',
    '  ICH E14/S7B 2022 — hERG / CiPA cardiac safety',
    '  EPA HTTK IVIVE   — in vitro to in vivo extrapolation',
    '  FDA Mod. Act 2.0 — NAM/ADMET data accepted in submissions',
    '  OECD GD 255      — IATA weight of evidence',
    '',
    'ERROR HANDLING RULES',
    '  Always return {"error": msg} never raise exceptions in call_tool()',
    '  Wrap entire call_tool body in try/except',
    '  Validate SMILES before any RDKit computation',
    '  Return structured JSON, never plain text',
    '',
    'TRANSPORT CHOICE',
    '  STDIO: local tools, single client, simplest setup',
    '  HTTP/SSE: team servers, network access, FastAPI-based',
]
print('\n'.join(lines))

In [ ]:
# ── Summary: all files created ───────────────────────────────────────────────
import os

files = [
    ('mcp_servers/hello_tox.py',           'Hello world MCP server (Section 2)'),
    ('mcp_servers/rdkit_tox_server.py',     'RDKit tools: ADMET, alerts, PAINS (Section 3)'),
    ('mcp_servers/tox_database_server.py',  'Toxicology DB as MCP resources (Section 4)'),
    ('mcp_servers/complete_admet_server.py','Production ADMET server — 6 tools (Section 7)'),
    ('claude_desktop_config.json',          'Claude Desktop config (Section 8)'),
    ('auto_tox_screen.py',                  'Anthropic API pipeline (Section 9)'),
]

print('Files created by this tutorial:')
print('='*65)
for path, desc in files:
    status = 'OK' if os.path.exists(path) else '--'
    print(f'  [{status}] {path:45s} {desc}')

print()
print('Quick start (3 steps):')
print('  1. pip install mcp rdkit')
print('  2. python mcp_servers/complete_admet_server.py  (runs the server)')
print('  3. Copy claude_desktop_config.json to Claude config dir, restart Claude')
print()
print('Then ask Claude:')
prompts = [
    'Run a full toxicology risk assessment on aspirin SMILES: CC(=O)Oc1ccccc1C(=O)O',
    'Is NDMA (CN(C)N=O) a genotoxic impurity? What ICH M7 class?',
    'Screen caffeine and diclofenac for hERG cardiac risk',
]
for p in prompts:
    print(f'  > {p}')